# Healthcare Chatbot for Google Colab

This notebook builds a simple healthcare chatbot in Python using public, no-key APIs:

- **NLM ClinicalTables** for condition and symptom search
- **RxNorm** for medication normalization
- **OpenFDA drug labels** for indications, warnings, side effects, and dosage text
- **Wikipedia REST summaries** as a fallback for general medical concepts

> ⚠️ **Important safety note:** No chatbot can answer health and medicine questions "without any mistake." This notebook is designed to provide educational information from public sources, not diagnosis or treatment. For emergencies, severe symptoms, pregnancy/child dosing, medication changes, or personal medical decisions, contact a licensed clinician or emergency services.


In [ ]:
import json
import re
import textwrap
import urllib.parse
import urllib.request
from html import unescape

USER_AGENT = 'ColabHealthcareChatbot/1.0 educational no-key API demo'
TIMEOUT = 12

print('✅ Libraries loaded. This notebook uses only the Python standard library.')


## Helper functions
These helpers call public APIs, clean responses, and apply basic medical safety guardrails.

In [ ]:
EMERGENCY_PATTERNS = [
    r'\b(chest pain|trouble breathing|shortness of breath|stroke|seizure|unconscious|not breathing)\b',
    r'\b(severe bleeding|suicidal|overdose|anaphylaxis|face drooping|weakness on one side)\b',
    r'\b(worst headache|severe allergic reaction|poisoning)\b',
]
HIGH_RISK_PATTERNS = [
    r'\b(diagnose me|what do i have|should i stop|should i start|dosage for me|dose for me)\b',
    r'\b(pregnant|pregnancy|infant|baby|child|kidney disease|liver disease)\b',
]

def wrap(text, width=92):
    return '\n'.join(textwrap.wrap(str(text), width=width, replace_whitespace=False))

def clean_text(value, max_chars=1200):
    if isinstance(value, list):
        value = ' '.join(str(x) for x in value if x)
    value = unescape(str(value or ''))
    value = re.sub(r'<[^>]+>', ' ', value)
    value = re.sub(r'\s+', ' ', value).strip()
    return value[:max_chars].rstrip()

def get_json(url):
    request = urllib.request.Request(url, headers={'User-Agent': USER_AGENT, 'Accept': 'application/json'})
    with urllib.request.urlopen(request, timeout=TIMEOUT) as response:
        return json.loads(response.read().decode('utf-8', errors='replace'))

def is_emergency(question):
    q = question.lower()
    return any(re.search(pattern, q) for pattern in EMERGENCY_PATTERNS)

def is_high_risk(question):
    q = question.lower()
    return any(re.search(pattern, q) for pattern in HIGH_RISK_PATTERNS)

def safety_prefix(question):
    if is_emergency(question):
        return ('🚨 This may be urgent. If this is happening now, call emergency services '
                '(911 in the U.S.) or your local emergency number immediately.\n\n')
    if is_high_risk(question):
        return ('⚠️ This question may require individualized medical advice. Use the information below '
                'for education only and confirm decisions with a licensed clinician.\n\n')
    return ('ℹ️ Educational information only; not a diagnosis or treatment plan.\n\n')

def extract_medication_term(question):
    """Heuristic cleanup for common medication questions like 'ibuprofen side effects'."""
    text = question.lower()
    text = re.sub(r'[^a-z0-9\- ]+', ' ', text)
    stop_phrases = [
        'what are', 'what is', 'tell me about', 'can you explain', 'side effects of',
        'side effect of', 'side effects', 'side effect', 'dosage of', 'dose of',
        'warnings for', 'warning for', 'uses of', 'use of', 'medicine', 'medication',
        'drug', 'pill', 'tablet', 'capsule', 'for me', 'for'
    ]
    for phrase in stop_phrases:
        text = text.replace(phrase, ' ')
    text = re.sub(r'\s+', ' ', text).strip()
    return text or question


## Public API search functions

In [ ]:
def search_clinical_tables(term, max_results=5):
    """Search NLM ClinicalTables Conditions API for health condition names."""
    params = urllib.parse.urlencode({'terms': term, 'maxList': max_results, 'df': 'primary_name,consumer_name'})
    url = 'https://clinicaltables.nlm.nih.gov/api/conditions/v3/search?' + params
    try:
        data = get_json(url)
        rows = data[3] if len(data) > 3 else []
        results = []
        for row in rows:
            name = row[0] if row else ''
            consumer = row[1] if len(row) > 1 else ''
            label = consumer or name
            if label and label not in results:
                results.append(label)
        return results
    except Exception as exc:
        return [f'ClinicalTables lookup unavailable: {exc}']

def rxnorm_lookup(term, max_results=5):
    """Find RxNorm medication concepts related to a term."""
    url = 'https://rxnav.nlm.nih.gov/REST/drugs.json?name=' + urllib.parse.quote(term)
    try:
        data = get_json(url)
        groups = data.get('drugGroup', {}).get('conceptGroup', [])
        meds = []
        for group in groups:
            for concept in group.get('conceptProperties', []) or []:
                name = concept.get('name')
                rxcui = concept.get('rxcui')
                if name and rxcui:
                    meds.append({'name': name, 'rxcui': rxcui})
                    if len(meds) >= max_results:
                        return meds
        return meds
    except Exception as exc:
        return [{'name': f'RxNorm lookup unavailable: {exc}', 'rxcui': ''}]

def openfda_label(term):
    """Fetch one OpenFDA drug label for common safety/usage sections."""
    query = urllib.parse.quote(f'openfda.generic_name:"{term}" OR openfda.brand_name:"{term}"')
    url = f'https://api.fda.gov/drug/label.json?search={query}&limit=1'
    try:
        data = get_json(url)
        if not data.get('results'):
            return None
        label = data['results'][0]
        sections = []
        for key, title in [
            ('indications_and_usage', 'Uses / indications'),
            ('warnings', 'Warnings'),
            ('adverse_reactions', 'Possible side effects'),
            ('dosage_and_administration', 'Dosage information from label'),
        ]:
            if key in label:
                sections.append(f"{title}: {clean_text(label[key], 700)}")
        return sections or None
    except Exception as exc:
        return [f'OpenFDA label lookup unavailable: {exc}']

def wikipedia_summary(term):
    """Fallback summary for general concepts."""
    title = urllib.parse.quote(term.strip().replace(' ', '_'))
    url = f'https://en.wikipedia.org/api/rest_v1/page/summary/{title}'
    try:
        data = get_json(url)
        extract = clean_text(data.get('extract'), 900)
        page = data.get('content_urls', {}).get('desktop', {}).get('page')
        if extract:
            return extract, page
    except Exception as exc:
        return f'Wikipedia lookup unavailable: {exc}', None
    return None, None


## Chatbot answer function

In [ ]:
def healthcare_chatbot(question):
    question = str(question).strip()
    if not question:
        return 'Please type a health or medicine question.'

    response = [safety_prefix(question)]
    q_lower = question.lower()

    # Medication-focused route when the wording suggests a drug question.
    med_match = re.search(r'\b(?:drug|medicine|medication|pill|tablet|capsule|side effects?|dose|dosage|warnings?|uses? of)\b(?:\s+for|\s+of)?\s+([a-zA-Z][a-zA-Z0-9\- ]{1,60})', q_lower)
    med_term = med_match.group(1).strip(' ?.!,') if med_match else extract_medication_term(question)
    rx_hits = rxnorm_lookup(med_term)
    label_sections = openfda_label(med_term)

    if rx_hits:
        valid_rx = [m for m in rx_hits if m.get('rxcui')]
        if valid_rx:
            response.append('Medication matches from RxNorm:')
            for med in valid_rx[:5]:
                response.append(f"- {med['name']} (RxCUI {med['rxcui']})")
    if label_sections:
        response.append('\nOpenFDA label information:')
        response.extend(f'- {section}' for section in label_sections[:4])

    # Condition/concept route.
    conditions = search_clinical_tables(question)
    usable_conditions = [c for c in conditions if not c.startswith('ClinicalTables lookup unavailable')]
    if usable_conditions:
        response.append('\nRelated health topics from NLM ClinicalTables:')
        response.extend(f'- {name}' for name in usable_conditions[:5])

    summary, page = wikipedia_summary(usable_conditions[0] if usable_conditions else question)
    if summary:
        response.append('\nGeneral overview:')
        response.append(summary)
        if page:
            response.append(f'Source: {page}')

    response.append('\nWhen to seek care: if symptoms are severe, worsening, new, or concerning, contact a healthcare professional. Never change prescribed medication without professional guidance.')
    return wrap('\n'.join(response))


## Ask one question
Run this cell and type a health question, for example: `What are symptoms of diabetes?` or `What are ibuprofen side effects?`

In [ ]:
question = input('Ask a health or medicine question: ')
print('\n' + healthcare_chatbot(question))


## Optional chat loop
Run this cell for an ongoing chat. Type `quit` to stop.

In [ ]:
while True:
    question = input('HealthBot> ').strip()
    if question.lower() in {'quit', 'exit', 'q'}:
        print('Goodbye. Stay safe!')
        break
    print('\n' + healthcare_chatbot(question) + '\n')
